# Data Preprocessing for Weather Dataset

This notebook replicates the data loading, cleaning, encoding, scaling, splitting and saving steps from the reference logistic regression tutorial. It stops before model training.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import numpy as np

In [4]:
# Load the raw CSV (the same file used in the tutorial)
raw_df = pd.read_csv("weatherAUS.csv")
raw_df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [5]:
# Drop rows where the target columns are missing (as suggested in the tutorial)
raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)
raw_df.shape

(140787, 23)

In [6]:
# Identify numeric and categorical columns
numeric_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = raw_df.select_dtypes(exclude=[np.number]).columns.tolist()
# Exclude the target column from encoding (it will stay as is)
categorical_cols = [c for c in categorical_cols if c not in ['RainTomorrow']]
numeric_cols = [c for c in numeric_cols if c not in ['RainTomorrow']]
numeric_cols, categorical_cols

(['MinTemp',
  'MaxTemp',
  'Rainfall',
  'Evaporation',
  'Sunshine',
  'WindGustSpeed',
  'WindSpeed9am',
  'WindSpeed3pm',
  'Humidity9am',
  'Humidity3pm',
  'Pressure9am',
  'Pressure3pm',
  'Cloud9am',
  'Cloud3pm',
  'Temp9am',
  'Temp3pm'],
 ['Date', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday'])

In [7]:
# Fill missing numeric values with median
for col in numeric_cols:
    raw_df[col].fillna(raw_df[col].median(), inplace=True)
# One‑hot encode categorical columns
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = encoder.fit_transform(raw_df[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)
cat_df = pd.DataFrame(cat_encoded, columns=cat_feature_names, index=raw_df.index)
# Scale numeric columns to [0,1]
scaler = MinMaxScaler()
num_scaled = scaler.fit_transform(raw_df[numeric_cols])
num_df = pd.DataFrame(num_scaled, columns=numeric_cols, index=raw_df.index)
# Combine all features with the target column
final_df = pd.concat([num_df, cat_df, raw_df[['RainTomorrow']]], axis=1)
final_df.head()

C:\Users\mst\AppData\Local\Temp\ipykernel_12876\312697594.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  raw_df[col].fillna(raw_df[col].median(), inplace=True)
C:\Users\mst\AppData\Local\Temp\ipykernel_12876\312697594.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

In [ ]:
# Split into train/validation/test sets (same proportions as the tutorial)
train_val_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df['RainTomorrow'])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df['RainTomorrow'])
print('Train:', train_df.shape, 'Validation:', val_df.shape, 'Test:', test_df.shape)

In [ ]:
# Save each split to CSV files
output_dir = './processed_weather'
import os
os.makedirs(output_dir, exist_ok=True)
train_df.to_csv(f'{output_dir}/train.csv', index=False)
val_df.to_csv(f'{output_dir}/validation.csv', index=False)
test_df.to_csv(f'{output_dir}/test.csv', index=False)
print('Datasets saved to', output_dir)